## 1. Preparación

Conecto Google Drive para cargar el dataset unificado y el vectorizador
TF-IDF que ya entrenamos en el modelo baseline. Reutilizo ese mismo
vectorizador para que los documentos guardados y
cualquier texto nuevo queden comparados en el mismo espacio numérico.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import joblib

RUTA_DATASET = '/content/drive/MyDrive/Datasets_TechMind/1 Semana - con el Dataset final con todo incluido - Repo GitHub/procesados/dataset_FINAL_UNIFICADO_techmind.csv'
RUTA_VECTORIZADOR = '/content/drive/MyDrive/Datasets_TechMind/modelo baseline/vectorizer.pkl'

df = pd.read_csv(RUTA_DATASET)
vectorizador = joblib.load(RUTA_VECTORIZADOR)

print("Filas del dataset:", len(df))
print("Vectorizador cargado, vocabulario de tamaño:", len(vectorizador.vocabulary_))

Mounted at /content/drive
Filas del dataset: 1400
Vectorizador cargado, vocabulario de tamaño: 3000


## 2. Convertir todos los documentos a números

Uso el vectorizador ya entrenado para convertir el texto limpio de los
1.400 documentos del dataset a su versión numérica (TF-IDF). Uso
transform(), no fit_transform() — no quiero que aprenda un vocabulario
nuevo, solo que use el que ya aprendió al entrenar el modelo.

In [3]:
import nltk
nltk.download('stopwords')

import sys
sys.path.append('/content/drive/MyDrive/Datasets_TechMind/similitud contenidos')

from limpieza_texto import limpiar_texto

print("Función importada correctamente")
print(limpiar_texto("¡Hola! Esto es una PRUEBA con HTML <b>y</b> números 123."))

df["texto_limpio"] = df["texto"].apply(limpiar_texto)

X_todos = vectorizador.transform(df["texto_limpio"])

print("Forma de la matriz completa (filas, columnas):", X_todos.shape)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Función importada correctamente
hola prueba html números 123
Forma de la matriz completa (filas, columnas): (1400, 3000)


## 3. Calcular similitud y encontrar los documentos más parecidos

Uso similitud coseno para comparar un texto nuevo contra los 1.400
documentos ya guardados. La similitud coseno mide qué tan parecida es la
"dirección" de dos vectores de números — cuanto más cercano a 1, más
parecidos son los textos; cuanto más cercano a 0, menos relación tienen.
Devuelvo los 3 documentos más parecidos al texto nuevo.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def encontrar_contenidos_relacionados(texto_nuevo, cantidad=3):
    texto_nuevo_limpio = limpiar_texto(texto_nuevo)
    vector_nuevo = vectorizador.transform([texto_nuevo_limpio])

    similitudes = cosine_similarity(vector_nuevo, X_todos)[0]

    indices_mas_similares = similitudes.argsort()[::-1][:cantidad]

    resultados = []
    for i in indices_mas_similares:
        resultados.append({
            "titulo": df.iloc[i]["titulo"],
            "categoria": df.iloc[i]["categoria"],
            "similitud": round(float(similitudes[i]), 3)
        })
    return resultados

## 4. Probar la función con textos de ejemplo

Pruebo con un texto sobre un tema puntual y reviso si los documentos que
devuelve como "relacionados" tienen sentido de verdad — mismo tipo de
chequeo manual que hicimos con la categoría del modelo baseline.

In [6]:
texto_prueba_1 = """
Kubernetes es una plataforma de orquestación de contenedores que automatiza
el despliegue, escalado y gestión de aplicaciones en contenedores Docker.
"""

resultados_1 = encontrar_contenidos_relacionados(texto_prueba_1)
for r in resultados_1:
    print(f"{r['similitud']} — [{r['categoria']}] {r['titulo']}")

0.589 — [Cloud] ¿Cuál es la diferencia entre Docker Compose y Kubernetes?
0.519 — [Cloud] Creación de aplicaciones en contenedores en AWS
0.409 — [DevOps] Docker Essentials y creación de una aplicación web en contenedores


In [7]:
texto_prueba_2 = """
React es una librería de JavaScript para construir interfaces de usuario
mediante componentes reutilizables. Permite crear aplicaciones web
interactivas usando un DOM virtual para mejorar el rendimiento.
"""

resultados_2 = encontrar_contenidos_relacionados(texto_prueba_2)
for r in resultados_2:
    print(f"{r['similitud']} — [{r['categoria']}] {r['titulo']}")

0.329 — [Frontend] Aprendizaje de React Native: creación de aplicaciones móviles nativas con JavaScript
0.327 — [Frontend] Desarrollo web front-end con React
0.299 — [Frontend] Comparación del rendimiento técnico del estudio de marcos frontend modernos en Svelte, React y Vue


## 5. Resumen

La función encuentra los 3 documentos más parecidos a un texto nuevo,
usando similitud coseno sobre el mismo vectorizador TF-IDF del modelo
baseline. Compara por contenido de texto, no por categoría — esto es a
propósito: permite encontrar relaciones útiles aunque crucen categorías
(por ejemplo, un texto sobre React puede relacionarse con contenido de
React Native, aunque una sea Frontend y la otra Mobile). La categoría de
cada resultado se muestra igual, para que quien lo use pueda decidir si
le interesa o no.
Lo probe con dos ejemplos de temas distintos (Docker/
Kubernetes y React)